# 🧹 Module 3 — Class 1: Clean the Telco Data

**Lesson:** [https://bepro-aiml.github.io/aiml-platform/#/module/3/class/1](https://bepro-aiml.github.io/aiml-platform/#/module/3/class/1)

---

## 📖 Today's Story

It is Monday morning. You work at a telecom company in Tashkent.

Your boss walks in:

> *"We are losing customers. Tell me who will leave next month. Use our customer data."*

You ask the data team for the file. They send you `customers.csv` and say:

> *"This file came from 3 different systems — billing, the website signup form, and the call-center logs. We merged them all together last week. Some fields are missing. Some values look weird. Some customers are in there twice. Sorry — that's what we have. Clean it before you use it."*

**This is real life.** Real customer data is always messy. Today we clean.

Tomorrow we train.

---

## 💡 Why is cleaning important?

**Bad data → bad model → wrong answer → angry boss.**

Real data scientists spend **80% of their time** cleaning data. Not training models. Cleaning.

If you can clean data well, you can do ML.

---

## 🤖 What is a notebook?

A notebook has two kinds of cells:
- 📝 **Text cells** (like this one) — they explain things.
- 💻 **Code cells** — Python code. You can run them.

**To run a code cell:** click on it. Press **Shift + Enter**.

## 📦 Read the comments!

Comments are lines that start with `#`. Python skips them. They are notes for YOU.

**Read every comment.** They tell you what each line does.

Let's go! 🚀

---
## Step 0: Get the Tools

### 💡 Why?

Python alone can not work with tables. We need helpers (libraries):
- **pandas** = tables
- **numpy** = numbers
- **matplotlib** + **seaborn** = charts

In [ ]:
# 'import' = bring in the tool. 'as' = give it a short name.
import pandas as pd               # tables
import numpy as np                # numbers
import matplotlib.pyplot as plt   # charts
import seaborn as sns             # nicer charts

import warnings
warnings.filterwarnings('ignore')

print("Tools ready!")

---
## Step 1: Get the Customer Data

We will use **Telco Customer Churn** data — about 7,000 real customers.

We load the original clean file from the internet, then **make it look like real production data** by adding the kinds of mistakes the data team's merge would create.

### 💡 Why fake the mess?

The Kaggle Telco file is too clean. Real-world data is **never** that clean. So we add real-world mistakes — missing values, typos, duplicates — to practice on something that looks like what you'll see at work.

In [ ]:
# Load the original (clean) Telco file from the internet.
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print(f"Loaded {df.shape[0]} customers, {df.shape[1]} columns.")

In [ ]:
# ─────────────────────────────────────────────
# Make the data look like real production data.
# This is the mess your data team would deliver.
# Don't worry about HOW we make the mess — focus
# on the cleaning steps that come after.
# ─────────────────────────────────────────────

# Same random seed = everyone gets the same mess.
np.random.seed(42)

# 1) Missing values in MonthlyCharges (90 random rows)
miss_rows = np.random.choice(df.index, size=90, replace=False)
df.loc[miss_rows, 'MonthlyCharges'] = np.nan

# 2) Missing values in tenure (50 random rows)
miss_rows2 = np.random.choice(df.index, size=50, replace=False)
df.loc[miss_rows2, 'tenure'] = np.nan

# 3) Inconsistent capitalization in Churn (Yes/yes/YES/ Yes )
yes_rows = df.index[df['Churn'] == 'Yes'].tolist()
np.random.shuffle(yes_rows)
df.loc[yes_rows[:30], 'Churn']  = 'yes'
df.loc[yes_rows[30:60], 'Churn'] = 'YES'
df.loc[yes_rows[60:90], 'Churn'] = ' Yes '

# 4) Impossible values (data entry errors)
df.loc[100, 'tenure']        = -5      # negative tenure
df.loc[200, 'tenure']        = 999     # 999 years? typo
df.loc[300, 'MonthlyCharges'] = 99999  # billing error or VIP?

# 5) Duplicate customers (the merge from 3 systems made copies)
df = pd.concat([df, df.iloc[[0, 50, 100, 500, 1000]]], ignore_index=True)

print(f"Now we have a 'real-life' messy file: {df.shape[0]} rows, {df.shape[1]} columns.")
df.head()

👀 **Look at the table above.**

Each row = one customer. Important columns:
- `tenure` = months as our customer
- `MonthlyCharges` = how much they pay each month
- `TotalCharges` = how much they paid all together
- `Churn` = did they leave? (Yes/No) ← **This is what we want to predict!**

It looks fine on the surface. But trust me — there are problems hiding inside. Let's find them.

---
## Step 2: Look Before You Touch

### 💡 Why?

Imagine you got a new car. **First thing?** You look around. Check the lights. Check the tires.

Same with data. **First we LOOK. Then we change.**

Three commands tell us a lot:

In [ ]:
# .info() shows: column name, type, and how many values are NOT empty.
#
# Look at 'Dtype':
#   'object'  = text
#   'int64'   = whole number (5)
#   'float64' = number with a dot (19.95)
df.info()

🚨 **Already 2 problems!**

1. **`TotalCharges`** says `object` (text). But it should be money — a number!
2. **`MonthlyCharges` and `tenure`** have fewer non-null values than total rows. Some are empty!

Let's check more:

In [ ]:
# .describe() = numbers about NUMBER columns.
# Always check min and max — bad data hides there!
df.describe()

🚨 **More problems!**

Look carefully:
- `tenure` min = **−5**. Negative months? Impossible!
- `tenure` max = **999**. 999 months = 83 years. Impossible!
- `MonthlyCharges` max = **99,999**. One customer pays 99,999 per month? Real or typo?

These are why we look BEFORE we touch.

In [ ]:
# How many empty values in each column?
# True = empty. .sum() counts the True values.
df.isnull().sum()

### 🤔 Stop and think

1. How many rows? How many columns?
2. Which column is `object` but should be a number?
3. Which 2 columns have empty values? About how many empty?
4. What is the smallest tenure? Why is that a problem?

---
## Step 3: Fix the Money Column 💰

### 💡 Why?

`TotalCharges` is text. **You can not do math on text!**

Try in your head: `"hello" + "world"` = `"helloworld"` (text)

But: `5 + 3` = `8` (number)

If the model sees `"100.50"` (text), it crashes.

**We must change text to numbers.** Why is it text? Let's find out!

In [ ]:
# Try to change to number. errors='coerce' = bad values become NaN.
# Then .isna() finds those bad values.
mask = pd.to_numeric(df['TotalCharges'], errors='coerce').isna()

print(f"Bad rows: {mask.sum()}")
print("\nLet's see them:")
df.loc[mask, ['customerID', 'TotalCharges', 'tenure', 'MonthlyCharges']].head(10)

💡 **AHA!** Look — most bad rows have `tenure = 0` (or NaN from our messy file).

These are **brand new customers**. They just signed up. They have not paid anything yet!

Their `TotalCharges` value is empty space (`" "`). That is why pandas read everything as text.

**Fix:**
1. Change to number column (empty becomes NaN).
2. New customers paid 0. So fill NaN with 0.

In [ ]:
# Step 1: Make it a number column. Bad values become NaN.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Step 2: Fill NaN with 0 (new customers paid 0).
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Check
print("Type now:", df['TotalCharges'].dtype)
print("Empty values:", df['TotalCharges'].isna().sum())

### 🧠 Important lesson

We filled with **0**, not the median. Why?

Because **0 is the correct answer here.** A new customer paid 0.

**Always think:** *what does the empty value MEAN?*

Sometimes empty = unknown. Sometimes empty = zero. Sometimes empty = a real signal.

👉 **Domain knowledge wins.** Talk to people who understand the business.

---
## Step 4: Fill the Empty Values

### 💡 Why?

Look — our file has **real empty values** in `MonthlyCharges` and `tenure`. Real life examples:
- A customer didn't fill in their monthly plan on the form.
- The signup system crashed for 1 hour.
- The merge from 3 systems lost some values.

**The model crashes with NaN. We must fill them.**

Three ways to fill:

| Way | When to use |
| --- | --- |
| **Mean** (average) | Numbers, no big outliers |
| **Median** (middle) | Numbers, safer with outliers |
| **Mode** (most common) | Text columns |

In [ ]:
# Check empty values now (we already saw them in Step 2).
print("Empty values per column:")
missing = df.isnull().sum()
print(missing[missing > 0])

In [ ]:
# Fill MonthlyCharges with the MEDIAN.
# Why median? It is safer if there are big outliers.
# (And we saw a 99,999 in Step 2, so median is the right choice.)
median_charges = df['MonthlyCharges'].median()
df['MonthlyCharges'] = df['MonthlyCharges'].fillna(median_charges)

# Fill tenure with the MEDIAN too (whole numbers — round to int after).
median_tenure = df['tenure'].median()
df['tenure'] = df['tenure'].fillna(median_tenure)

print(f"Median MonthlyCharges: {median_charges:.2f}")
print(f"Median tenure: {median_tenure}")
print(f"\nEmpty values now: {df.isnull().sum().sum()}")

### 📝 YOUR TURN — add a 'missing indicator' column

Sometimes 'this value is empty' tells us something useful!
- A customer who didn't fill in `MonthlyCharges` → maybe didn't trust the form → maybe will leave.
- The fact of empty IS information.

**But we just filled them all!** So we lost that info.

Better way: **add a 'missing' column BEFORE you fill.**

### 📚 Hint

```python
df['MonthlyCharges_missing'] = df['MonthlyCharges'].isna().astype(int)
# .astype(int) changes True/False to 1/0
```

But we already filled `MonthlyCharges` — so isna() returns all False now.

**This is a real lesson:** ORDER MATTERS in cleaning. Missing-indicators must come first.

In [ ]:
# YOUR CODE — make a missing indicator column for tenure_was_missing or MonthlyCharges_was_missing
# (Try it on a fresh copy from Step 1 if you want to do it 'before fillna')

# Hint:
# df['tenure_was_missing'] = df['tenure'].isna().astype(int)

---
## Step 5: Same Word, Same Meaning

### 💡 Why?

Look at the `Churn` column. Real production data has these problems:
- Some rows: `'Yes'`
- Some rows: `'yes'` (different signup form)
- Some rows: `'YES'` (auto-caps from a phone keyboard)
- Some rows: `' Yes '` (extra spaces from Excel)

**The model thinks they are 4 different categories!** But they all mean the same thing.

Same problem in service columns: `'No'`, `'No internet service'`, `'No phone service'` — all mean *the customer doesn't have it*.

**One meaning = one word.** Always.

In [ ]:
# Look at Churn — is it really messy?
print("Churn values:")
print(df['Churn'].value_counts())

🚨 See? **5 different ways** to say Yes/No.

**Fix:** strip spaces, then make all letters the same (lowercase or capitalize).

In [ ]:
# Two-step clean for text:
#   .str.strip()       = remove spaces at start/end
#   .str.capitalize()  = first letter big, rest small ('Yes', 'No')
df['Churn'] = df['Churn'].str.strip().str.capitalize()

print("Churn values after clean:")
print(df['Churn'].value_counts())

In [ ]:
# Now the service columns. They have 'No internet service' / 'No phone service'.
# These really mean 'No'.
internet_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in internet_cols:
    df[col] = df[col].replace('No internet service', 'No')

df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

# Check
print("OnlineSecurity values now:", df['OnlineSecurity'].unique())
print("MultipleLines values now :", df['MultipleLines'].unique())

✅ Now the model sees only 2 categories. Much better!

---
## Step 6: Remove Duplicate Customers

### 💡 Why?

When the data team merged 3 systems, the same customer can appear **twice** (or more)!

If we count Ali twice, our metrics are wrong:
- We think we have 7,048 customers. We have 7,043.
- The model trains on Ali's data twice = it 'memorizes' Ali too well.

**Each customer should appear ONCE.**

In [ ]:
# How many duplicate customers?
# subset=['customerID'] = same customer ID = duplicate
dups = df.duplicated(subset=['customerID']).sum()
print(f"Duplicate customers: {dups}")
print(f"Total rows now: {len(df)}")

# Drop duplicates. Keep the first one.
df = df.drop_duplicates(subset=['customerID']).reset_index(drop=True)

print(f"\nAfter dedupe: {len(df)} rows.")

---
## Step 7: Find the Weird Numbers (Outliers)

### 💡 Why?

Remember from Step 2:
- `tenure = -5` → **impossible** (negative months)
- `tenure = 999` → **impossible** (83 years!)
- `MonthlyCharges = 99,999` → typo? VIP? fraud?

Three kinds of outliers:

1. **Mistakes** (typing errors, broken sensors) → fix or remove
2. **Real but very high** (a VIP customer) → keep, maybe cap to a max
3. **The thing you want to find** (fraud!) → never remove!

🚨 **Always think first: which kind do I have?**

In [ ]:
# Step 1: Fix the IMPOSSIBLE values.
# tenure must be 0 or more, and not absurdly high.
# We replace impossible values with NaN, then fill with the median.

impossible = (df['tenure'] < 0) | (df['tenure'] > 200)
print(f"Impossible tenure values: {impossible.sum()}")
print(f"They are: {df.loc[impossible, 'tenure'].tolist()}")

df.loc[impossible, 'tenure'] = np.nan
df['tenure'] = df['tenure'].fillna(df['tenure'].median())

In [ ]:
# Step 2: Find statistical outliers using the IQR rule.
#
# Q1   = 25% of values are below this
# Q3   = 75% of values are below this
# IQR  = Q3 - Q1
# anything outside [Q1 - 1.5×IQR, Q3 + 1.5×IQR] = outlier

def detect_outliers_iqr(series, factor=1.5):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    return (series < lower) | (series > upper), lower, upper

for col in ['MonthlyCharges', 'TotalCharges', 'tenure']:
    outliers, low, high = detect_outliers_iqr(df[col])
    print(f"{col}: {outliers.sum()} outliers (limits: [{low:.2f}, {high:.2f}])")

💡 We found a real outlier in `MonthlyCharges` — the **99,999** value!

Three choices:
1. **Remove it** — easy but lose the customer
2. **Cap (winsorize)** — set it to the upper limit. Keep the customer, prevent extreme value from breaking things
3. **Keep it** — if it's real (a VIP)

We pick **cap** — safer:

In [ ]:
# Cap MonthlyCharges to the IQR upper limit.
# .clip(lower, upper) replaces below-lower with lower, above-upper with upper.
outliers, low, high = detect_outliers_iqr(df['MonthlyCharges'])
df['MonthlyCharges'] = df['MonthlyCharges'].clip(lower=low, upper=high)

print(f"MonthlyCharges max now: {df['MonthlyCharges'].max():.2f}")

In [ ]:
# Box plots — the BEST chart to see distribution + outliers.
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate(['MonthlyCharges', 'TotalCharges', 'tenure']):
    axes[i].boxplot(df[col].dropna())
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

### 🤔 Stop and think

Some of `tenure` might still show dots (outliers). But 72 months = 6 years is just a loyal customer.

👉 **A 6-year customer is REAL!** We do NOT remove them. Their pattern teaches the model what loyal customers look like!

**Lesson:** *Big number ≠ wrong number.* Always think first.

---
## Step 8: Save the Clean Data

### 💡 Why?

Tomorrow we use this clean file to **train the model**.

If we don't save, we have to clean again. Waste of time!

**Rule:** Never write over the first file. Always save with a NEW name.

In [ ]:
# Last check
print("Shape:", df.shape)
print("Empty values:", df.isnull().sum().sum())
print("\nColumn types:")
print(df.dtypes.value_counts())

In [ ]:
# Save to a NEW file. index=False = don't save row numbers.
df.to_csv('telco_churn_cleaned.csv', index=False)
print("✅ Saved: telco_churn_cleaned.csv")

# In Colab — download to your computer:
# from google.colab import files
# files.download('telco_churn_cleaned.csv')

---
## 🏁 We Did It!

### What we cleaned today

| Step | What we found | What we did |
| --- | --- | --- |
| 2. Look | TotalCharges as text, missing values, impossible numbers | Listed problems |
| 3. Fix money | 11 blank-string rows in TotalCharges | Made it a number, filled with 0 |
| 4. Fill empty | 90 missing in MonthlyCharges, 50 in tenure | Filled with median |
| 5. Same words | 'yes', 'YES', ' Yes ', 'No internet service' | One spelling per category |
| 6. Duplicates | 5 customers in twice from the merge | Dropped duplicates |
| 7. Outliers | tenure=-5, tenure=999, MonthlyCharges=99,999 | Replaced + capped |
| 8. Save | Clean file ready | Saved as `telco_churn_cleaned.csv` |

### 🎯 Where are we going?

**Today:** Clean the data ✅

**Class 2:** Make the data ready for the model (encoding, scaling).

**Class 3:** Build new useful columns (feature engineering).

**Module 4:** Train the model. Predict who will leave!

Each class makes the next class possible. 🪜

### 🎓 Words to Remember

- **DataFrame** — a table
- **NaN** — empty value
- **dtype** — type of a column
- **Median** — middle value (safer than mean for dirty data)
- **Outlier** — a value very different from the others
- **Winsorize / clip** — cap a value to a max instead of removing it
- **Domain knowledge** — knowing the business, not just the math

### 📤 Submit

Save this notebook as `Module3_Class1_<YourName>.ipynb`.

Push to your group repo at `module-3/class_1/submissions/<YourName>/`.

🧹 *Great job! See you in Class 2!*